# Economic Analysis: Education, Wealth & Brain Drain Study

**Course:** CSE130 | **Group:** 5
**Team Members:** Maimuna Rahman, Jannatul Mahabuba Adhora, MD. Sharafatul Islam Shihab, Zinna Tun Nahar Medha, Zarin Subah

## Executive Summary
This notebook explores the statistical relationships between education, economic development, billionaire wealth distribution, and international brain drain. The analysis is divided into four parts, examining global datasets (183 countries) and a 51-year longitudinal study of Bangladesh.

**Data Sources:** World Bank, Forbes, UNDP, ILO, and MacroTrends.

In [ ]:
# Import necessary data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling for professional academic plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the datasets
# Ensure 'world.csv' and 'bangladesh.csv' are in a 'csv_files/' folder relative to this notebook
try:
    df_world = pd.read_csv('csv_files/world.csv')
    df_bd = pd.read_csv('csv_files/bangladesh.csv')
    print("Datasets loaded successfully!")
    print(f"World Dataset: {df_world.shape[0]} countries.")
    print(f"Bangladesh Dataset: {df_bd.shape[0]} years.")
except FileNotFoundError as e:
    print(f"Error loading data: {e}. Please check your file paths.")

# Display the first few rows to verify structure
display(df_world.head(3))

## Analysis 1: Per Capita Income Regression
**Research Question:** Do more educated countries earn significantly higher incomes?
**Methodology:** Bivariate linear regression between average years of education and average annual salary (USD).

In [ ]:
# Drop missing values for this specific analysis
data_a1 = df_world[['avg_education', 'avg_salary']].dropna()

x = data_a1['avg_education']
y = data_a1['avg_salary']

# Calculate linear regression and Pearson correlation
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

print(f"Regression Equation: Salary = {slope:,.0f} × Education + ({intercept:,.0f})")
print(f"Pearson Correlation (r): {r_value:.4f}")

# Plotting
sns.regplot(x=x, y=y, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.title('Analysis 1: Education vs Average Salary')
plt.xlabel('Average Years of Education')
plt.ylabel('Average Annual Salary (USD)')
plt.show()

## Analysis 2: The Billionaire Effect on National Averages
**Research Question:** How much do billionaires artificially inflate the "average" per capita income of a nation?
**Methodology:** Comparative analysis using the $Q3 + 3 \times IQR$ method to identify extreme outliers and contrast income with/without billionaires.

In [ ]:
# Recreate the 3x IQR logic
q1 = df_world['avg_salary'].quantile(0.25)
q3 = df_world['avg_salary'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + (3 * iqr)

print(f"Extreme Outlier Threshold (3x IQR): ${upper_bound:,.2f}")

# Boxplot comparison (recreating analysis2_boxplot.png)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(y=df_world['income_with_billionaire'], color='skyblue')
plt.title('Income (With Billionaires)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df_world['income_without_billionaire'], color='lightgreen')
plt.title('Income (Without Billionaires)')
plt.tight_layout()
plt.show()

# Calculate the inflation percentage
global_avg_with = df_world['income_with_billionaire'].mean()
global_avg_without = df_world['income_without_billionaire'].mean()
inflation_pct = ((global_avg_with - global_avg_without) / global_avg_without) * 100
print(f"Billionaires inflate the global average income by {inflation_pct:.2f}%")

## Analysis 3: The Talent Retention Gap
**Research Question:** Does a country's ability to produce billionaires correlate with its ability to retain overall talent?
**Methodology:** Bivariate correlation comparing Billionaire Density to Brain Drain metrics.

In [ ]:
# Calculate required metrics as they are not pre-computed in the CSV
df_world['billionaire_density'] = df_world['billionaire_born'] / df_world['population_with_billionaire']
df_world['brain_drain_index'] = df_world['billionaire_left'] / df_world['billionaire_born']

# Filter out where billionaire_born is 0 to avoid division by zero
data_a3 = df_world[df_world['billionaire_born'] > 0].copy()

x_a3 = data_a3['billionaire_density']
y_a3 = data_a3['brain_drain_index']

r_val_a3, _ = stats.pearsonr(x_a3.dropna(), y_a3.dropna())
print(f"Correlation between Billionaire Density and Brain Drain: r = {r_val_a3:.4f}")

sns.scatterplot(x=x_a3, y=y_a3, alpha=0.7, color='purple')
plt.title('Analysis 3: Brain Drain vs Billionaire Density')
plt.xlabel('Billionaires per Million People')
plt.ylabel('Brain Drain Metric (Emigrated / Born)')
plt.show()

## Analysis 4: The Bangladesh Dilemma
**Research Question:** As Bangladesh's economy grows, does the emigration of students decrease?
**Methodology:** Time-series analysis and correlation over 51 years (1975–2025) between GDP growth and student outflow.

In [ ]:
# Adjusting column names to match actual CSV headers
year = df_bd['Year']
gdp = df_bd['GDP_Per_Capita_USD']
students_out = df_bd['S_out']

r_val_bd, _ = stats.pearsonr(gdp, students_out)
print(f"Correlation (GDP vs Student Emigration): r = {r_val_bd:.4f}")

# Recreating the dual-axis chart (analysis4_bangladesh.png)
fig, ax1 = plt.subplots()

color = 'tab:blue'
ax1.set_xlabel('Year')
ax1.set_ylabel('GDP per Capita (USD)', color=color)
ax1.plot(year, gdp, color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Students Emigrating', color=color)
ax2.plot(year, students_out, color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Analysis 4: GDP Growth vs. Student Emigration in Bangladesh (1975-2025)')
fig.tight_layout()  
plt.show()

## Key Conclusions

* **The Education Dividend:** Education moderately predicts national income ($r = 0.6526$), adding approximately $5,410 to the average salary per extra year of schooling.
* **The Billionaire Paradox:** Despite public perception, billionaires inflate the global average income mathematically by only 1.75%.
* **The Talent Retention Gap:** Producing extreme wealth does not guarantee talent retention ($r = -0.2050$). Small nations lose top talent, while developed economies retain it based on structural environment.
* **The Bangladesh Dilemma:** Over 51 years, economic growth and emigration operated independently ($r = 0.1650$). Growth provided the financial means for students to leave rather than incentivizing them to stay.